# Fractional Laplacian: Nonlocal Diffusion and Lévy Processes

The classical Laplacian $\Delta = \partial_{xx} + \partial_{yy}$ governs local diffusion: the value of $\Delta f$ at a point depends only on $f$ in an infinitesimal neighborhood. The **fractional Laplacian** $(-\Delta)^s$ for $s \in (0,1)$ is a **nonlocal** operator where the action at any point integrates contributions from the entire domain.

## Fourier definition

The fractional Laplacian is defined by its Fourier symbol:
$$
\widehat{(-\Delta)^s f}(\xi) = |2\pi\xi|^{2s}\, \hat{f}(\xi).
$$
For integer $s = 1$, this reduces to $-\Delta$ (Fourier symbol $|2\pi\xi|^2$). For $s < 1$, the power law gives **polynomial decay** of the spectral multiplier, while $s > 1$ amplifies high frequencies faster.

## Integral representation

For $s \in (0,1)$, the fractional Laplacian has a **principal value integral** representation:
$$
(-\Delta)^s f(x) = C_{n,s} \, \text{P.V.}\int_{\mathbb{R}^n} \frac{f(x) - f(y)}{|x-y|^{n+2s}}\, dy,
$$
where $C_{n,s}$ is a normalization constant. This shows the **nonlocal nature**: the operator computes a weighted average of differences $f(x) - f(y)$ over all $y$, with the kernel $|x-y|^{-(n+2s)}$ decaying as a power law.

## Physical interpretations

- **$s < 1/2$**: ultra-slow diffusion (stable Lévy process with heavy jump tails).
- **$s = 1/2$**: Cauchy process; harmonic extension to a half-space.
- **$s \to 1$**: classical Brownian diffusion.
- **$s > 1$**: hyper-diffusion, smoothing faster than Brownian motion.

## Heat semigroup

The **fractional heat equation** $\partial_t u + (-\Delta)^s u = 0$ generates a semigroup $e^{-t(-\Delta)^s}$, the fractional heat kernel, which has Fourier symbol $e^{-t|\xi|^{2s}}$ and decays as a power law in space for $s < 1$.

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

plt.rcParams['figure.dpi'] = 120

## 1D fractional Laplacian via FFT

On the periodic interval $[0, 1]$ with $n$ grid points, the fractional Laplacian is diagonal in Fourier space with eigenvalue $|2\pi k|^{2s}$ for frequency $k$.

In [ ]:
n = 2048
t = np.linspace(0, 1, n, endpoint=False)
om = np.fft.fftfreq(n) * n  # integer frequencies

def frac_laplacian_1d(f, s):
    """(-Delta)^s in 1D on periodic grid."""
    F = np.fft.fft(f)
    lam = (2 * np.pi * np.abs(om)) ** (2*s)
    lam[0] = 0  # zero mean / DC
    return np.real(np.fft.ifft(F * lam))

def frac_heat_1d(f, t_time, s):
    """Solution of partial_t u + (-Delta)^s u = 0 at time t."""
    F = np.fft.fft(f)
    decay = np.exp(-t_time * (2*np.pi*np.abs(om))**(2*s))
    return np.real(np.fft.ifft(F * decay))

print('Fractional Laplacian ready.')

## Effect of $s$ on the operator

We apply $(-\Delta)^s$ to a smooth bump for varying $s$. The result shows how the operator changes character: for small $s$ the output is broad and smooth; for $s$ near 1 it resembles $-\Delta$ (sharp, localized).

In [ ]:
# Input: smooth bump
f_bump = np.exp(-((t - 0.5)**2) / (2 * 0.04**2))
f_bump -= f_bump.mean()  # zero mean

s_values = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
cols = plt.cm.plasma(np.linspace(0.1, 0.9, len(s_values)))

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, s, col in zip(axes.ravel(), s_values, cols):
    Lf = frac_laplacian_1d(f_bump, s)
    Lf_norm = Lf / (np.max(np.abs(Lf)) + 1e-14)
    x_zoom = np.abs(t - 0.5) < 0.35
    ax.plot(t[x_zoom], f_bump[x_zoom]/f_bump.max(), 'gray', lw=1.2, alpha=0.5, ls='--', label='$f$')
    ax.plot(t[x_zoom], Lf_norm[x_zoom], lw=2.5, color=col, label=fr'$(-\Delta)^{{{s}}} f$')
    ax.axhline(0, color='k', lw=0.8, ls=':')
    ax.set_title(fr'$s = {s}$', fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_ylim(-1.4, 1.4)
fig.suptitle(r'Fractional Laplacian $(-\Delta)^s$ of a Gaussian bump for varying $s$', y=1.02)
plt.tight_layout()
plt.show()

## Fractional heat equation: anomalous diffusion

The heat semigroup $u(x,t) = e^{-t(-\Delta)^s} f$ describes diffusion at rate determined by $s$. For $s < 1$, the tails of the solution are heavy (power-law), corresponding to **Lévy flights** — a model for anomalous diffusion in disordered media and financial returns.

In [ ]:
# Initial condition: Dirac-like delta
f_delta = np.zeros(n); f_delta[n//2] = 1.0
f_delta -= f_delta.mean()

t_times = [0.0001, 0.001, 0.01, 0.1]
s_compare = [0.3, 0.7, 1.0]
cols_s = plt.cm.tab10([0, 1, 2])

fig, axes = plt.subplots(len(s_compare), len(t_times), figsize=(13, 9))
for row, (s, col) in enumerate(zip(s_compare, cols_s)):
    for col_idx, t_t in enumerate(t_times):
        u = frac_heat_1d(f_delta, t_t, s)
        u_norm = u / (u.max() + 1e-14)
        zoom = np.abs(t - 0.5) < 0.4
        axes[row, col_idx].plot(t[zoom], u_norm[zoom], lw=2, color=col)
        axes[row, col_idx].set_ylim(-0.1, 1.1)
        axes[row, col_idx].grid(alpha=0.3)
        if row == 0:
            axes[row, col_idx].set_title(f'$t = {t_t}$', fontsize=9)
        if col_idx == 0:
            axes[row, col_idx].set_ylabel(fr'$s = {s}$', fontsize=10)
fig.suptitle(r'Fractional heat $e^{-t(-\Delta)^s}$: heavy tails for $s < 1$', y=1.02)
plt.tight_layout()
plt.show()

## Spectral multiplier and eigenvalues

The eigenfunctions of $(-\Delta)^s$ on the periodic interval are the Fourier modes $e^{2\pi i k x}$, with eigenvalues $(2\pi |k|)^{2s}$. Plotting the eigenvalue spectrum reveals the power-law growth that distinguishes fractional from local diffusion.

In [ ]:
k_pos = np.arange(1, 100)
fig, ax = plt.subplots(figsize=(8, 4.5))
for s, col in zip([0.25, 0.5, 0.75, 1.0, 1.5],
                   plt.cm.plasma(np.linspace(0.1, 0.9, 5))):
    eigs = (2*np.pi*k_pos)**(2*s)
    ax.loglog(k_pos, eigs, lw=2.5, color=col, label=fr'$s={s}$')
ax.set_xlabel('frequency $k$'); ax.set_ylabel('eigenvalue $(2\\pi k)^{2s}$')
ax.set_title('Fractional Laplacian eigenvalues: power law $k^{2s}$')
ax.legend(fontsize=9); ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## Interactive: sweep $s$ and time

Observe how the fractional diffusion kernel spreads in real space as $t$ grows, and how the tail heaviness depends on $s$.

In [ ]:
def show_frac_lap(s=0.5, log_t=-2.0):
    t_time = 10**log_t
    u = frac_heat_1d(f_delta, t_time, s)
    u_norm = u / (u.max() + 1e-14)
    zoom = np.abs(t - 0.5) < 0.45

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(t[zoom], u_norm[zoom], lw=2.5, color='royalblue')
    axes[0].set_xlabel('$x$'); axes[0].set_ylabel('$u(x,t)$ (normalized)')
    axes[0].set_title(fr'$(-\Delta)^{{{s:.2f}}}$ heat kernel at $t={t_time:.4f}$')
    axes[0].grid(alpha=0.3)

    F_u = np.abs(np.fft.fft(u)[:n//4])
    k_range = np.arange(n//4)
    axes[1].semilogy(k_range[1:200], F_u[1:200] / (F_u[1]+1e-14), 'b-', lw=2)
    axes[1].set_xlabel('frequency $k$'); axes[1].set_ylabel('$|\\hat{u}(k)|$')
    axes[1].set_title('Fourier spectrum'); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

interact(show_frac_lap,
         s=FloatSlider(value=0.5, min=0.05, max=1.5, step=0.05, description='$s$'),
         log_t=FloatSlider(value=-2.0, min=-4.0, max=0.0, step=0.25,
                           description='$\\log_{10} t$'));

## Bibliographical resources

- Caffarelli, L. and Silvestre, L. (2007). An extension problem related to the fractional Laplacian. *Communications in Partial Differential Equations*, 32(8), 1245–1260.
- Kwaśnicki, M. (2017). Ten equivalent definitions of the fractional Laplace operator. *Fractional Calculus and Applied Analysis*, 20(1), 7–51.
- Applebaum, D. (2009). *Lévy Processes and Stochastic Calculus* (2nd ed.). Cambridge University Press.
- Bucur, C. and Valdinoci, E. (2016). *Nonlocal Diffusion and Applications*. Springer.
- Lischke, A. et al. (2020). What is the fractional Laplacian? A comparative review with new results. *Journal of Computational Physics*, 404, 109009.